# Token-level方案评估：完整grad是否必要 & 如何高效实现

本Notebook回答两个问题：
1. 你的实验是否**必须**收集完整grad向量？
2. 在需要时，如何以低开销方式做 token-level 分析（mixed 与 vision-only 都适用）

> 新版日志支持 `grad_path`（推荐）与 `grad`（兼容）。默认只存统计量，按需为少量关键参数落盘完整向量。


In [ ]:
import ast, json
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style='whitegrid', context='talk')


In [ ]:
LOG_PATH = Path('gradient_logs.jsonl')
OUT = Path('analysis_outputs')
OUT.mkdir(parents=True, exist_ok=True)

STEP_MIN=None
STEP_MAX=None
STEP_MOD=None
MAX_ROWS=250000
MAX_VEC_ROWS=80000
MAX_PAIR=30000

assert LOG_PATH.exists(), LOG_PATH
print('using', LOG_PATH.resolve())


In [ ]:
def _step_ok(v, lo=None, hi=None, mod=None):
    try: s=int(v)
    except: return False
    if lo is not None and s<lo: return False
    if hi is not None and s>hi: return False
    if mod is not None and mod>1 and s%mod!=0: return False
    return True

def load_subset(path, usecols=None, partition_in=None, max_rows=None):
    usecols=set(usecols) if usecols else None
    rows=[]
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line=line.strip()
            if not line: continue
            try: o=json.loads(line)
            except: continue
            if 'step' in o and not _step_ok(o['step'], STEP_MIN, STEP_MAX, STEP_MOD):
                continue
            if partition_in is not None and o.get('grad_partition','all') not in partition_in:
                continue
            rows.append(o if usecols is None else {k:o.get(k,None) for k in usecols})
            if max_rows and len(rows)>=max_rows: break
    df=pd.DataFrame(rows)
    if len(df)==0: return df
    for c in ['step','layer_depth','grad_norm','grad_norm_per_token','supervised_token_count','partition_token_ratio','text_token_ratio','image_token_ratio']:
        if c in df.columns: df[c]=pd.to_numeric(df[c],errors='coerce')
    if 'grad_partition' in df.columns: df['grad_partition']=df['grad_partition'].fillna('all')
    if 'grad_was_none' in df.columns: df['grad_was_none']=df['grad_was_none'].fillna(False).astype(bool)
    return df


## 1) 必要性评估：什么时候必须收集完整grad？

In [ ]:
meta = load_subset(LOG_PATH, usecols=[
    'step','grad_partition','layer_depth','grad_norm','grad_norm_per_token','grad_was_none',
    'batch_mix_type','partition_token_ratio','text_token_ratio','image_token_ratio',
    'grad_path','grad'
], max_rows=MAX_ROWS)

if len(meta)==0:
    raise ValueError('No rows loaded; check filters')

has_grad_path = ('grad_path' in meta.columns) and meta['grad_path'].notna().any()
has_grad_inline = ('grad' in meta.columns) and meta['grad'].notna().any()

need_full_grad_reasons = []
need_full_grad_reasons.append('需要做余弦方向冲突（cos）')
need_full_grad_reasons.append('需要做子空间容量SVD/rank95')
need_full_grad_reasons.append('需要做跨层梯度投影/对齐诊断')

print('has_grad_path:', has_grad_path)
print('has_inline_grad:', has_grad_inline)
print('结论：若只做趋势/范数统计，可不收集完整grad；若做方向与子空间机制分析，必须收集完整grad。')
print('建议：默认统计日志 + Top-K参数 grad_path 按步采样（当前代码已支持）')


## 2) token-level基础分析（不依赖完整grad）

In [ ]:
if 'grad_norm_per_token' not in meta.columns or meta['grad_norm_per_token'].notna().mean()==0:
    if {'grad_norm','supervised_token_count'}.issubset(meta.columns):
        meta['grad_norm_per_token'] = meta['grad_norm'] / meta['supervised_token_count'].clip(lower=1)

plt.figure(figsize=(12,5))
trend = meta.groupby(['step','grad_partition'], as_index=False)['grad_norm_per_token'].mean()
sns.lineplot(data=trend, x='step', y='grad_norm_per_token', hue='grad_partition')
plt.title('Mean grad_norm_per_token over steps')
plt.tight_layout(); plt.savefig(OUT/'fig_token_trend_partition.png', dpi=180); plt.show()


In [ ]:
if 'batch_mix_type' not in meta.columns or meta['batch_mix_type'].notna().mean()==0:
    meta['batch_mix_type'] = 'unknown'

mixed = meta[meta['batch_mix_type']=='text_and_vision']
vision = meta[meta['batch_mix_type']=='vision_only']
print('mixed rows:', len(mixed), 'vision rows:', len(vision))

if len(mixed)>0:
    key=['step','layer_depth']
    w=mixed.pivot_table(index=key, columns='grad_partition', values='grad_norm_per_token', aggfunc='mean').reset_index()
    if {'text_only','image_only'}.issubset(w.columns):
        eps=1e-12
        w['dom']=(w['image_only']-w['text_only'])/(w['image_only']+w['text_only']+eps)
        plt.figure(figsize=(10,4)); sns.histplot(w['dom'], bins=60, kde=True); plt.axvline(0,color='r',ls='--')
        plt.title('Dominance (mixed batches)'); plt.tight_layout(); plt.savefig(OUT/'fig_mixed_dom_hist.png', dpi=180); plt.show()


## 3) 向量加载（支持 grad_path 与 grad）

In [ ]:
def parse_inline_grad(g):
    if g is None:
        return None
    if isinstance(g,(list,tuple,np.ndarray)):
        return np.asarray(g,dtype=np.float32).reshape(-1)
    if isinstance(g,str):
        s=g.strip()
        if not s: return None
        try:
            x=json.loads(s)
            if isinstance(x,list): return np.asarray(x,dtype=np.float32).reshape(-1)
        except: pass
        try:
            x=ast.literal_eval(s)
            if isinstance(x,(list,tuple)): return np.asarray(x,dtype=np.float32).reshape(-1)
        except: pass
    return None

def load_grad_vector(row):
    gp = row.get('grad_path', None)
    if isinstance(gp, str) and gp and Path(gp).exists():
        return np.load(gp).reshape(-1)
    return parse_inline_grad(row.get('grad', None))

vec = load_subset(
    LOG_PATH,
    usecols=['step','param_name','layer_depth','adapter_type','grad_partition','batch_mix_type','grad_path','grad'],
    partition_in={'text_only','image_only','all'},
    max_rows=MAX_VEC_ROWS,
)

if len(vec)==0:
    print('no vector rows')
else:
    vec['vector'] = vec.apply(load_grad_vector, axis=1)
    vec = vec[vec['vector'].notna()].copy()
    print('usable vector rows:', len(vec))


## 4) 方向冲突（cos）与容量（rank95）

In [ ]:
def cosine(a,b,eps=1e-12):
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+eps))

def rank95(X):
    if X.shape[0] < 2: return np.nan
    X = X - X.mean(0, keepdims=True)
    s = np.linalg.svd(X, full_matrices=False, compute_uv=False)
    e = np.cumsum(s*s) / (np.sum(s*s)+1e-12)
    return int(np.searchsorted(e, 0.95)+1)

if len(vec)>0:
    key=['step','param_name','layer_depth','adapter_type']
    if 'batch_mix_type' in vec.columns: key.append('batch_mix_type')
    t=vec[vec['grad_partition']=='text_only'][key+['vector']].rename(columns={'vector':'tv'})
    i=vec[vec['grad_partition']=='image_only'][key+['vector']].rename(columns={'vector':'iv'})
    pair=t.merge(i,on=key,how='inner').head(MAX_PAIR)

    cos_vals=[]
    for _,r in pair.iterrows():
        a,b=r['tv'],r['iv']
        m=min(len(a),len(b))
        cos_vals.append(np.nan if m==0 else cosine(a[:m],b[:m]))
    pair['cos']=cos_vals
    pair=pair.dropna(subset=['cos'])

    plt.figure(figsize=(9,4)); sns.histplot(pair['cos'], bins=60, kde=True); plt.axvline(0,color='r',ls='--')
    plt.title('Cos(text,image)'); plt.tight_layout(); plt.savefig(OUT/'fig_cos_text_image.png', dpi=180); plt.show()

    if 'batch_mix_type' in pair.columns:
        plt.figure(figsize=(10,4)); sns.boxplot(data=pair, x='batch_mix_type', y='cos')
        plt.title('Cos by composition'); plt.tight_layout(); plt.savefig(OUT/'fig_cos_by_composition.png', dpi=180); plt.show()

    rec=[]
    gcols=['layer_depth','grad_partition'] + (['batch_mix_type'] if 'batch_mix_type' in vec.columns else [])
    for gk,g in vec.groupby(gcols):
        gg=g.head(180)
        if len(gg)<2: continue
        d=min(len(v) for v in gg['vector'])
        if d<=1: continue
        X=np.stack([v[:d] for v in gg['vector']], axis=0)
        rec.append((*((gk,) if not isinstance(gk,tuple) else gk), len(gg), d, rank95(X)))

    if len(rec)>0:
        cols=gcols+['n','dim','rank95']
        rdf=pd.DataFrame(rec, columns=cols)
        rdf.to_csv(OUT/'table_rank95_token_level.csv', index=False, encoding='utf-8-sig')
        plt.figure(figsize=(11,5))
        if 'batch_mix_type' in rdf.columns:
            sns.lineplot(data=rdf,x='layer_depth',y='rank95',hue='grad_partition',style='batch_mix_type',markers=True,dashes=False)
        else:
            sns.lineplot(data=rdf,x='layer_depth',y='rank95',hue='grad_partition',marker='o')
        plt.title('Rank95 by layer/partition/composition')
        plt.tight_layout(); plt.savefig(OUT/'fig_rank95_token_level.png', dpi=180); plt.show()


## 5) 实验方案结论（回答你的问题）

- **完整grad不是默认必需**：若你只做趋势/范数/占比统计，token-level统计字段足够。  
- **完整grad在机制研究中必需**：一旦要证明“方向冲突（cos）”与“子空间容量不足（SVD）”，必须有向量。  
- **最佳实践（已落地到代码）**：默认仅统计日志，按 Top-K 参数写 `grad_path`，既保留机制证据又控制开销。  
